# ENG-QUANT quick start

Choose one of the two workflows below:

- **toy**: build a small quadratic toy problem, create a linear approximation, and solve it.
- **real**: linearize an existing JSON problem, remap its nonzero lower bounds to zero, and solve it.

Run this notebook in the same directory as your JSON files.

In [ ]:
%load_ext autoreload
%autoreload 2

In [23]:
from eng_quant.generators import toy_problem_to_json_reduced
from eng_quant.relaxers import (
    quadratic_lin_approx_no_surrogate,
    remap_lower_bounds
)
from eng_quant.ipm import (
    ipm_solve,
    ipm_solve_fo,
    ipm_solve_so,
    ipm_solve_no_split,
    ipm_solve_no_split_fo,
    ipm_solve_no_split_so
)



## Input options

Set `PROBLEM_KIND` to `"toy"` or `"real"`. For a real problem, set `REAL_PROBLEM_FILE` to a JSON file in `ENG-QUANT-NEW/experiments`.

In [24]:
# Workflow: "toy" or "real"
PROBLEM_KIND = "toy"

# Toy-problem inputs
N = 1
F_UPPER_BOUNDS = True
DEMAND_INEQUALITY = False
GAMMA = 1.0
LAMBD = 1.0
EPS = 0.1
DELTA = 0.1
LP = LF = 0.0
UP = 0.5
CONVEX_RELAX = True
TOY_OUTPUT = "toy_quickstart.json"

# Real-problem inputs
REAL_PROBLEM_FILE = "model11_quad_reform.json"

# Linear approximation inputs
LIN_CONSTRAINTS = 10
OUTER_APPROXIMATION = True
ENDPOINTS = True
REMOVE_DIVISION = False

if PROBLEM_KIND not in {"toy", "real"}:
    raise ValueError("PROBLEM_KIND must be 'toy' or 'real'.")

In [ ]:
# Build a toy problem or select a real problem.
if PROBLEM_KIND == "toy":
    toy_problem_to_json_reduced(
        N,
        F_UPPER_BOUNDS,
        DEMAND_INEQUALITY,
        GAMMA,
        LAMBD,
        EPS,
        DELTA,
        LP,
        LF,
        UP,
        convex_relax=CONVEX_RELAX,
        f_out=TOY_OUTPUT,
        verbose=True,
    )
    quadratic_input = TOY_OUTPUT
else:
    quadratic_input = REAL_PROBLEM_FILE

# Convert quadratic constraints into a linear-approximation JSON file.
quadratic_lin_approx_no_surrogate(
    LIN_CONSTRAINTS,
    OUTER_APPROXIMATION,
    remove_division=REMOVE_DIVISION,
    endpoints=ENDPOINTS,
    f_name=quadratic_input,
    verbose=True,
)

# Real models commonly contain nonzero lower bounds. The IPM requires 0.
if PROBLEM_KIND == "real":
    remap_lower_bounds(
        f_name="linear_approx.json",
        verbose=True,
        quadratic_check=True,
        lb=0,
    )

Variables: 5
Non-bound constraints: 5
Approximation points for quadratic variable f[production1_transient1]:
[0.         0.11111111 0.22222222 0.33333333 0.44444444 0.55555556
 0.66666667 0.77777778 0.88888889 1.        ]
Approximation points for quadratic variable f[production2_transient1]:
[0.         0.07856742 0.15713484 0.23570226 0.31426968 0.3928371
 0.47140452 0.54997194 0.62853936 0.70710678]
Approximation points for quadratic variable f[transient1_customer1]:
[0.         0.07856742 0.15713484 0.23570226 0.31426968 0.3928371
 0.47140452 0.54997194 0.62853936 0.70710678]
Made a linear approximation with 32 (non-bound) constraints.


## IPM options

Set `SOLVER` to `"split"`, `"split_fo"`, `"split_so"`, `"no_split"`, `"no_split_fo"`, or `"no_split_so"`.

Start with `qsim=False` for the classical path where supported.

In [ ]:
# Solver: "split", "split_fo", "split_so", "no_split", "no_split_fo", or "no_split_so"
SOLVER = "split_fo"

SOLVERS = {"split", "split_fo", "split_so", "no_split", "no_split_fo", "no_split_so"}
if SOLVER not in SOLVERS:
    raise ValueError(f"SOLVER must be one of {sorted(SOLVERS)}.")

IPM_INPUT = "linear_approx.json"
IPM_BETA = 0.1
IPM_XI = 1 - 5e-8
IPM_OMEGA = 1e4
IPM_EPSILON = 1e-8
IPM_INFEAS_FACTOR = 1000.0
SYMMETRIC_NEIGHBORHOOD = True
ZETA_N = 1e-2
ZETA_R = 0.3
ALPHA_SCHED = [0.9, 0.7, 0.5, 0.3, 0.1, 0.01]
ALPHA_MIN = 1e-3
SAUND_TOM = False
SAUND_TOM_FACTOR = 1e-16
SYMMETRIC_SCALING = True
OMEGA_BOUND = True
SPARSE = False
CLASSIC_ILU = False
QSIM = False
REFINEMENT_ITERS = 10
IR_RTOL = 1e-10
ILU_DROP_TOL = 1e-4
BACKEND_NAME = "emulated"
DALZELL_INVERSION = False
HADAMARD_READOUT = False
IPM_VERBOSE = True
PRINT_QLSA_STATS = True
DRAW_QLSA = False
PHYS_EST_QLSA = False

In [ ]:
base_ipm_options = {
    "f_name": IPM_INPUT,
    "xi": IPM_XI,
    "omega": IPM_OMEGA,
    "epsilon": IPM_EPSILON,
    "infeas_factor": IPM_INFEAS_FACTOR,
    "sym_neighborhood": SYMMETRIC_NEIGHBORHOOD,
    "zeta_n": ZETA_N,
    "zeta_r": ZETA_R,
    "alpha_sched": ALPHA_SCHED,
    "alpha_min": ALPHA_MIN,
    "saund_tom": SAUND_TOM,
    "saund_tom_factor": SAUND_TOM_FACTOR,
    "omega_bound": OMEGA_BOUND,
    "symmetric_scaling": SYMMETRIC_SCALING,
    "sparse": SPARSE,
    "classic_ilu": CLASSIC_ILU,
    "refinement_iters": REFINEMENT_ITERS,
    "ir_rtol": IR_RTOL,
    "ilu_drop_tol": ILU_DROP_TOL,
    "verbose": IPM_VERBOSE,
}

solver_functions = {
    "split": ipm_solve,
    "split_fo": ipm_solve_fo,
    "split_so": ipm_solve_so,
    "no_split": ipm_solve_no_split,
    "no_split_fo": ipm_solve_no_split_fo,
    "no_split_so": ipm_solve_no_split_so,
}

solver_options = {}
if SOLVER in {"split", "no_split"}:
    solver_options["beta"] = IPM_BETA

if SOLVER in {"split", "split_fo", "no_split", "no_split_fo"}:
    solver_options.update({
        "qsim": QSIM,
        "backend_name": BACKEND_NAME,
        "dalzell_inversion": DALZELL_INVERSION,
        "hadamard_readout": HADAMARD_READOUT,
        "print_qlsa_stats": PRINT_QLSA_STATS,
        "draw_qlsa": DRAW_QLSA,
        "phys_est_qlsa": PHYS_EST_QLSA,
    })

objective, run_time, solution, complementarity_gap = solver_functions[SOLVER](
    **base_ipm_options, **solver_options
)
print(f"\nFinal result ({SOLVER})")
print(f"  Objective: {objective:.10f}")
print(f"  Runtime: {run_time:.3f} s")
print(f"  Solution: [{', '.join(f'{value:.8f}' for value in solution)}]")
print(f"  Absolute complementarity gap: {complementarity_gap:.3e}")

Restricted license - for non-production use only - expires 2028-11-29


Time to load matrices: 0.01 seconds.
Solution is below target precision.
The algorithm stopped after 27 iterations in 0.01 seconds.

Primal objective:        -1.10958590e+00
Dual objective:          -1.10958591e+00

Norm of Primal residual: 2.09e-11
Norm of Dual residual:   6.94e-12
Abs. Compl. Gap:         8.12e-09
Rel. Compl. Gap:         2.52e-09


Final result (split_fo)
  Objective: 1.1095859056
  Runtime: 0.008 s
  Solution: [0.25308641, 0.50000000, 0.00414093, 0.59585906, 0.50414094]
  Absolute complementarity gap: 8.115e-09
